# Détection de campagnes de désinformation

Projet pédagogique AI for Good avec données imaginées.

Objectif : entraîner un modèle de machine learning qui distingue des publications `organic` de publications ressemblant à une campagne coordonnée `campaign`.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


## 1. Créer un petit dataset fictif

Dans un vrai projet, il faudrait des données bien documentées, vérifiées, anonymisées et évaluées éthiquement. Ici, tout est imaginé.


In [ ]:
data = [
    ["Je viens de lire un rapport officiel, il faut vérifier les sources avant de partager.", 730, 850, 2, 0, 0.05, "organic"],
    ["URGENT!!! Tous les médias cachent la vérité, partagez avant suppression!!!", 12, 35, 48, 1, 0.93, "campaign"],
    ["Analyse intéressante, mais je vais attendre d'autres confirmations.", 420, 1200, 3, 1, 0.10, "organic"],
    ["Ils ne veulent pas que vous sachiez ceci, copiez-collez ce message partout maintenant.", 5, 22, 62, 1, 0.96, "campaign"],
    ["Résumé du débat municipal : plusieurs candidats ont proposé des mesures différentes.", 980, 340, 1, 0, 0.02, "organic"],
    ["BREAKING: preuve secrète révélée, les experts mentent tous, cliquez ici.", 18, 70, 54, 1, 0.91, "campaign"],
    ["J'ai comparé trois articles et les chiffres ne disent pas tous la même chose.", 640, 410, 4, 0, 0.08, "organic"],
    ["Le même message circule dans tous les groupes : partagez massivement sans poser de questions.", 9, 44, 73, 1, 0.98, "campaign"],
    ["Voici une infographie avec les sources citées et les limites de la méthode.", 350, 980, 5, 1, 0.12, "organic"],
    ["La vérité explosive que les autorités veulent effacer aujourd'hui même.", 7, 25, 81, 1, 0.95, "campaign"],
    ["Je ne suis pas certaine de cette information, quelqu'un a-t-il une source fiable?", 560, 260, 2, 0, 0.04, "organic"],
    ["Copiez ce texte exactement et publiez-le à 20h pour contourner la censure.", 3, 15, 95, 0, 0.99, "campaign"],
    ["Article long mais nuancé sur les risques et les bénéfices possibles.", 810, 1500, 1, 1, 0.03, "organic"],
    ["Tous les comptes doivent publier ce slogan dans les 10 prochaines minutes.", 14, 60, 67, 0, 0.92, "campaign"],
    ["J'ai trouvé les données publiques, elles montrent une tendance graduelle.", 900, 760, 3, 1, 0.06, "organic"],
    ["Ne vérifiez rien, les preuves sont déjà partout, partagez vite.", 6, 31, 88, 1, 0.94, "campaign"],
]
columns = ["text", "account_age_days", "followers", "posts_last_24h", "has_url", "duplicate_similarity", "label"]
df = pd.DataFrame(data, columns=columns)
df.head()


## 2. Préparer le modèle

On combine le texte avec des métadonnées simples.


In [ ]:
X = df.drop(columns=["label"])
y = df["label"]

numeric_features = ["account_age_days", "followers", "posts_last_24h", "has_url", "duplicate_similarity"]

preprocessor = ColumnTransformer([
    ("text", TfidfVectorizer(lowercase=True, ngram_range=(1, 2)), "text"),
    ("meta", StandardScaler(), numeric_features),
])

model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])


## 3. Entraîner et évaluer


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

print(classification_report(y_test, pred))
cm = confusion_matrix(y_test, pred, labels=["organic", "campaign"])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["organic", "campaign"])
disp.plot()
plt.title("Matrice de confusion")
plt.show()


## 4. Tester une nouvelle publication


In [ ]:
new_posts = pd.DataFrame([
    {
        "text": "Copiez ce message partout, ils veulent cacher la vérité demain matin.",
        "account_age_days": 8,
        "followers": 30,
        "posts_last_24h": 80,
        "has_url": 1,
        "duplicate_similarity": 0.95,
    },
    {
        "text": "Je cherche une source indépendante avant de partager cette nouvelle.",
        "account_age_days": 700,
        "followers": 500,
        "posts_last_24h": 2,
        "has_url": 0,
        "duplicate_similarity": 0.04,
    }
])

predictions = model.predict(new_posts)
probas = model.predict_proba(new_posts)
campaign_index = list(model.classes_).index("campaign")

for i, row in new_posts.iterrows():
    print(row["text"])
    print("Prédiction :", predictions[i])
    print("Score campagne :", round(probas[i][campaign_index] * 100, 2), "%")
    print()


## 5. Idées pour améliorer le projet

- Ajouter plus de données.
- Ajouter une détection de hashtags coordonnés.
- Créer une interface Gradio.
- Ajouter des explications du modèle.
- Utiliser CamemBERT pour mieux traiter le français.
